# NB01 - Data audit, extraction and label definition
Reads the 12 OASIS-1 disc archives **once**, extracts only the processed brain-masked scans (+ tissue segmentation) to fast local disk,
joins them with the clinical table and **freezes the labels**.

* Primary task: CDR = 0 vs CDR > 0, age >= 60 &nbsp;|&nbsp; Secondary: CDR 0 / 0.5 / >= 1
* One scan per participant (MR1). MR2 repeat scans are excluded and logged.
* MMSE, education and SES are never model inputs.
* Younger participants (< 60) are kept aside for out-of-distribution tests only.

Run time: about 15-40 min the first time (reading 12 archives), seconds afterwards.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
manifest, rep = PL.step1_audit(cfg, paths)
from oasis_ad.utils import _clean
import json; print(json.dumps(_clean({k: rep[k] for k in ['cohort_flow','role_counts','label_counts_binary','label_counts_3class','three_class_warning','young_ood_n']}), indent=2, default=str))

### Age confounding - is age a shortcut?

In [ ]:
from oasis_ad.utils import _clean
print(json.dumps(_clean(rep['age_confounding']), indent=2, default=str))
from IPython.display import Image, display
display(Image(f"{paths.audit}/age_confounding.png"))

### Check list before continuing
* Cohort flow numbers look plausible (about 436 sessions, about 235 with CDR at age >= 60)
* `three_class_warning` - if the CDR >= 1 group is small, report 3-class results descriptively only
* Everything is saved in `OASIS1_outputs/step1_audit/`

In [ ]:
manifest[manifest.role=='labeled'][['participant_id','age','sex','cdr','label_binary','label_3class','nwbv','etiv']].head(10)